# Watermarking for Summarization
**Laizhen Li · Shenzhen Institutes of Advanced Technology, Chinese Academy of Sciences**

Implementation and analysis of Soft, Hard, and Stratified Watermarking on BART-large-CNN and CNN/DailyMail.

Run the bootstrap cell to download the fixed GitHub revision. Then reconstruct archived results on CPU, or explicitly enable a new GPU experiment. Archived results are not newly generated outputs. See `VERIFICATION.json` for tested environments.

Reference: [A Watermark for Large Language Models](https://arxiv.org/abs/2301.10226).


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, io, urllib.request

REPOSITORY = "Lilaizhen/stratified-watermarking-summarization"
REVISION = "PINNED_COMMIT_SHA"  # Set to the published code commit before sharing.
if not (Path("src/workflow.py").exists() and Path("archived/records.json").exists()):
    if "YOUR_GITHUB_USERNAME" in REPOSITORY or REVISION == "PINNED_COMMIT_SHA":
        raise ValueError("Publication configuration is pending: set repository and pinned commit.")
    if len(REVISION) != 40 or any(c not in "0123456789abcdef" for c in REVISION.lower()):
        raise ValueError("Use a full 40-character commit SHA for reproducibility.")
    destination = Path("watermark_" + REVISION).resolve()
    marker = destination / ".download_complete"
    if not marker.exists():
        url = f"https://codeload.github.com/{REPOSITORY}/zip/{REVISION}"
        with urllib.request.urlopen(url, timeout=120) as response:
            payload = response.read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for member in archive.infolist():
                relative = Path(*Path(member.filename).parts[1:])
                target = (destination / relative).resolve()
                if not target.is_relative_to(destination):
                    raise ValueError("Unsafe archive path")
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(archive.read(member))
        marker.write_text(REVISION)
    os.chdir(destination)
sys.path.insert(0, str(Path("src").resolve()))
print("Project:", Path.cwd())
print("Code revision:", REVISION)


## 1. Environment and integrity
Run this in a fresh runtime. Select a GPU only for the new experiment. Installing pinned PyTorch may require a runtime restart; after restarting, rerun from the first cell. Use `INSTALL=False` only in an already prepared environment.

Core versions are pinned to the local experiment. Colab images change; a conflicting optional torchvision/torchaudio installation may need matching to torch 2.6.0. Neither package is required here.

In [ ]:
import hashlib
manifest = json.loads(Path("MANIFEST.json").read_text())
for name, digest in manifest.items():
    content = Path(name).read_bytes()
    if name == "configs/focused/alignscore.json":
        value = json.loads(content)
        for key in ("checkpoint", "backbone_snapshot"):
            value.pop(key, None)
        content = (json.dumps(value, indent=2) + "\n").encode()
    if hashlib.sha256(content).hexdigest() != digest:
        raise ValueError(f"Package differs: {name}")
print("Verified", len(manifest), "files")
INSTALL = True
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Protocol and fixed data splits
- BART-large-CNN, pinned checkpoint; CNN/DailyMail 3.0.0 validation.
- Temperature **0.8**, full-vocabulary multinomial sampling (**top-k=0, top-p=1**).
- **Temperature first, then watermark bias** for Soft and Stratified.
- Main comparison: γ=0.5; Stratified ρ=0.98; δ=0.5,1.0,…,6.0,8,10,12.
- **200 calibration articles**, disjoint from **500 evaluation articles**. All configurations share evaluation IDs.
- No extra special-token probability protection in Stratified. Hard exempts the forced initial BOS step; normal generation constraints are retained.
- FP16 generation, batch size 8, seed 1729 plus batch starting offset; source limit 1024, output 16–192 new tokens. Changing batch size changes random-number grouping.

`main`: 32 configurations (baseline, Hard, 15 Soft, 15 Stratified). `full`: 66 configurations, adding γ and ρ sensitivity analyses. The probability-mass diagnostic is a separate pass.

In [ ]:
base = json.loads(Path("configs/base.json").read_text())
splits = json.loads(Path("configs/splits.json").read_text())
assert len(splits["calibration"]) == 200 and len(splits["test"]) == 500
assert not {r["id"] for r in splits["calibration"]} & {r["id"] for r in splits["test"]}
print(base)
print({k: len(v) for k, v in splits.items()})

## 3. Watermarking methods
Soft assigns $q(i) \propto p(i)e^{\delta 1[i\in G]}$, where $p$ is the temperature-scaled distribution. Hard masks red logits to $-\infty$.

Stratified partitions $p$ into the smallest descending-probability head reaching ρ (including the crossing token) and its complement. Within each region $B$:

$$q(i)=P_B\frac{p(i)e^{\delta 1[i\in G]}}{\sum_{j\in B}p(j)e^{\delta 1[j\in G]}},\quad P_B=\sum_{j\in B}p(j).$$

This preserves each region's total probability at the current prefix, not individual token probabilities. The detector reconstructs the same keyed colors; it does not need the partition. The implementation below uses equivalent log-sum-exp corrections. Passing `special_ids=[]` implements pure stratification.

In [ ]:
import inspect
from headmass_processor import partition, mass_preserving_tilt
print(inspect.getsource(partition))
print(inspect.getsource(mass_preserving_tilt))
subprocess.run([sys.executable, "tests/check_methods.py"], check=True)

## 4. Reconstruct the saved report results (no GPU)
These outputs use archived experiments, not newly generated summaries. The reconstruction checks article IDs, source hashes across evaluation stages, independent calibration, and corpus PPL from per-summary negative log-likelihoods.

For each target TPR, select the lowest-PPL configuration satisfying it, with smaller δ breaking ties. Actual TPRs need not be equal. Table entries and plotted points are measured configurations; connecting lines are visual guides.

In [ ]:
from IPython.display import display, Image
from plots import rebuild
metrics, targets, gamma = rebuild()
columns = ["name", "delta", "tpr", "ppl", "rouge1_fmeasure", "rouge2_fmeasure", "rougeL_fmeasure", "alignscore_nli_sp"]
display(metrics[metrics.name.isin(["nw", "hard"])][columns])
display(targets[["target"] + columns])
display(gamma[["gamma", "threshold", "tpr", "fpr", "auc", "ppl", "rougeL_fmeasure", "alignscore_nli_sp"]])
for figure in ["bias_detection", "bias_quality", "head_mass", "rho_detection", "quality_detection"]:
    display(Image(filename=f"outputs/archived/{figure}.png"))

## 5. Detection and independent calibration
For distinct eligible adjacent token pairs, $z=(N_G-gT)/\sqrt{Tg(1-g)}$, where $g=\lfloor\gamma|V|\rfloor/|V|$. Transitions involving special tokens are excluded. Repeated pairs count once. Summaries are decoded and re-tokenized before detection; unscorable summaries remain non-detections and use −∞ for ROC ranking.

The threshold is the ascending $\lceil(n+1)(1-0.01)\rceil$-th calibration score, and detection uses **z > τ**. Evaluation negatives are used to measure FPR and AUC, not to choose τ. A separate threshold is calibrated for each γ. The report's γ=0.5 threshold is approximately 2.672612; evaluation FPR is 0.6%.

Color reconstruction uses CUDA RNG to match the experiment, but does not run a language model.

In [ ]:
from analysis import threshold
import numpy as np
calibration = json.loads(Path("archived/calibration.json").read_text())["0.5"]["scores"]
negatives = json.loads(Path("archived/negatives.json").read_text())["0.5"]["scores"]
to_scores = lambda records: np.array([r["z"] if r["z"] is not None else -np.inf for r in records])
tau = threshold(to_scores(calibration))
print("Calibration n:", len(calibration), "threshold:", tau)
print("Evaluation n:", len(negatives), "measured FPR:", np.mean(to_scores(negatives) > tau))

## 6. Generate and evaluate new outputs (GPU)
Set `RUN_NEW=True`. The default demo evaluates 16 articles and retains the independent 200-article calibration set. For a fast plumbing check, reduce `N_CALIBRATION`, but such a run cannot estimate report-level detection reliably.

`main` and `full` force the complete 200/500 split. The full grid requires 33,000 evaluation summaries plus 200 calibration summaries and may exceed a free Colab session. Persist `OUTPUT` to Drive if needed. Runtime/VRAM depend on GPU and enabled evaluators; no hosted timing guarantee is made.

Generation resumes from the last complete batch. Completed evaluation stages are reused only if source hashes match. The same output directory rejects changed settings. Models run in separate subprocesses to release GPU memory between stages.

Quality metrics: ROUGE-1/2/L F1 (Lin, 2004), unconditional token-weighted GPT-2-medium PPL (Radford et al., 2019), and AlignScore-base `nli_sp` source consistency (Zha et al., 2023). PPL excludes BOS/EOS/padding targets. AlignScore uses the documented local inference port; it is not a human factual-correctness rate. `QUALITY=False` skips PPL and AlignScore explicitly.

In [ ]:
RUN_NEW = False
MODE = "demo"  # demo, main, full
N_DEMO = 16
N_CALIBRATION = 200
QUALITY = True
OUTPUT = "runs/demo"  # Use a persistent mounted Drive directory for long runs.
if RUN_NEW:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime")
    print(torch.cuda.get_device_name(), torch.__version__)
    args = [sys.executable, "src/setup_assets.py"] + (["--quality"] if QUALITY else [])
    subprocess.run(args, check=True)
    from workflow import run
    new_metrics = run(mode=MODE, n=N_DEMO, n_calibration=N_CALIBRATION, quality=QUALITY, output=OUTPUT)
    display(new_metrics)
    if QUALITY:
        from analysis import select_targets
        display(select_targets(new_metrics))

## 7. Inspect summaries and distribution changes
Compare source, reference, and paired outputs directly. The optional diagnostic teacher-forces saved unwatermarked prefixes, fixes each original head token set, and measures its total probability after Soft at δ=0,…,6 (step 0.5). This is separate from generation and detection; it averages over positions, not articles. Positions within an article are correlated.

Use all 500 articles for the report diagnostic. Demo diagnostics use only the generated demo articles. Output directories are immutable for the diagnostic; choose a new one to rerun.

In [ ]:
RUN_DIAGNOSTIC = False
if RUN_NEW:
    for directory in sorted(Path(OUTPUT).iterdir()):
        file = directory / "generations.jsonl"
        if not file.exists() or directory.name == "unwatermarked_calibration":
            continue
        with file.open() as stream:
            row = json.loads(next(stream))
        print("\n", directory.name, "\n", row["summary"])
        if directory.name == "nw":
            print("SOURCE:", row["article"][:1200], "\nREFERENCE:", row["reference"])
    if RUN_DIAGNOSTIC:
        subprocess.run([sys.executable, "src/diagnostic.py", "--source", str(Path(OUTPUT)/"nw"),
                        "--output", str(Path(OUTPUT)/"head_diagnostic"), "--articles", str(N_DEMO if MODE=="demo" else 500),
                        "--soft-order", "post", "--soft-only"], check=True)

## 8. Export and verification scope
New run folders contain configurations, article IDs, generations, per-summary evaluation records and aggregate metrics. Export only your new run, not cached model weights.

Single-seed results and automatic metrics do not establish statistical significance, attack robustness, or general superiority. GPU/library changes can alter sampling. See `VERIFICATION.json` for checks actually completed; local cached-model tests are distinct from fresh hosted Colab execution.

Metric references: [ROUGE](https://aclanthology.org/W04-1013/), [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf), [AlignScore](https://aclanthology.org/2023.acl-long.634/). Full methodological references are in the accompanying report.

In [ ]:
if RUN_NEW:
    import shutil
    archive = shutil.make_archive("new_experiment", "zip", OUTPUT)
    print("Exported", archive)
    # from google.colab import files
    # files.download(archive)
print(json.loads(Path("VERIFICATION.json").read_text()))